# 라이브러리 추가 사항

아래 내용을 추가해주세요

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))

from src.evaluation import (
    evaluate_all,
    init_history,
    update_history,
    save_history,
    load_history,
    plot_training_history,
    plot_compare_histories,
    convert_yolo_results,
    convert_torchvision_outputs,
)

# 훈련 루프에 평가 지표 넣는 내용

훈련 루프 안에 아래 내용을 맞춰서 넣으시면 됩니다. 

모두 내용이 달라서 큰 틀과 예시 코드를 드립니다.

In [ ]:
history = init_history()

for epoch in range(1, <변경: num_epochs> + 1):
    # 1. 한 epoch 학습
    train_loss = <변경: 해당 epoch의 train 평균 loss>

    # 2. 한 epoch 검증
    val_loss = <변경: 해당 epoch의 val 평균 loss>

    # 3. 검증셋 예측 결과를 공통 형식으로 변환
    val_predictions = <변경: 공통 prediction format으로 변환된 검증셋 예측 결과>

    # 4. 평가
    metrics = evaluate_all(
        gt_json_path=<변경: 검증용 COCO json 경로>,
        predictions=val_predictions,
        conf_threshold=0.25,
        pr_iou_threshold=0.5,
        temp_json_path=f"<변경: 모델명>_temp_eval_epoch_{epoch}.json"
    )

    # 5. 기록
    update_history(
        history,
        epoch=epoch,
        train_loss=train_loss,
        val_loss=val_loss,
        metrics=metrics
    )

# 6. 저장 및 시각화
save_history(history, "<변경: 모델명별 history 파일명>.json")
plot_training_history(history, title_prefix="<변경: 모델명>")

# yolo 예시

In [ ]:
history = init_history()

for epoch in range(1, num_epochs + 1):
    # 1. train
    train_loss = epoch_train_loss

    # 2. val
    val_loss = epoch_val_loss

    # 3. 검증셋 예측
    # val_image_paths: 검증 이미지 경로 리스트
    # val_image_ids: COCO의 image_id 리스트
    results = model.predict(val_image_paths, conf=0.001, verbose=False)

    val_predictions = convert_yolo_results(results, val_image_ids)

    # 4. 평가
    metrics = evaluate_all(
        gt_json_path=VAL_JSON_PATH,
        predictions=val_predictions,
        conf_threshold=0.25,
        pr_iou_threshold=0.5,
        temp_json_path=f"yolo_temp_eval_epoch_{epoch}.json",
        model2orig=model2orig
    )

    # 5. 기록
    update_history(
        history,
        epoch=epoch,
        train_loss=train_loss,
        val_loss=val_loss,
        metrics=metrics
    )

save_history(history, "history_yolo.json")
plot_training_history(history, title_prefix="YOLO")

# Retina 예시

In [ ]:
history = init_history()

for epoch in range(1, num_epochs + 1):
    # 1. train
    train_loss = epoch_train_loss

    # 2. val
    val_loss = epoch_val_loss

    # 3. 검증셋 예측
    all_outputs = []
    all_image_ids = []

    model.eval()
    with torch.no_grad():
        for images, targets in val_loader:
            images = [img.to(device) for img in images]
            outputs = model(images)

            batch_image_ids = [t["image_id"].item() for t in targets]

            all_outputs.extend(outputs)
            all_image_ids.extend(batch_image_ids)

    val_predictions = convert_torchvision_outputs(all_outputs, all_image_ids)

    # 4. 평가
    metrics = evaluate_all(
        gt_json_path=VAL_JSON_PATH,
        predictions=val_predictions,
        conf_threshold=0.25,
        pr_iou_threshold=0.5,
        temp_json_path=f"retinanet_temp_eval_epoch_{epoch}.json",
        model2orig=model2orig
    )

    # 5. 기록
    update_history(
        history,
        epoch=epoch,
        train_loss=train_loss,
        val_loss=val_loss,
        metrics=metrics
    )

save_history(history, "history_retinanet.json")
plot_training_history(history, title_prefix="RetinaNet")

# Faster R-CNN 예시

In [ ]:
history = init_history()

for epoch in range(1, num_epochs + 1):
    # 1. train
    train_loss = epoch_train_loss

    # 2. val
    val_loss = epoch_val_loss

    # 3. 검증셋 예측
    all_outputs = []
    all_image_ids = []

    model.eval()
    with torch.no_grad():
        for images, targets in val_loader:
            images = [img.to(device) for img in images]
            outputs = model(images)

            batch_image_ids = [t["image_id"].item() for t in targets]

            all_outputs.extend(outputs)
            all_image_ids.extend(batch_image_ids)

    val_predictions = convert_torchvision_outputs(all_outputs, all_image_ids)

    # 4. 평가
    metrics = evaluate_all(
        gt_json_path=VAL_JSON_PATH,
        predictions=val_predictions,
        conf_threshold=0.25,
        pr_iou_threshold=0.5,
        temp_json_path=f"faster_rcnn_temp_eval_epoch_{epoch}.json",
        model2orig=model2orig
    )

    # 5. 기록
    update_history(
        history,
        epoch=epoch,
        train_loss=train_loss,
        val_loss=val_loss,
        metrics=metrics
    )

save_history(history, "history_faster_rcnn.json")
plot_training_history(history, title_prefix="Faster R-CNN")

# 평가지표 시각화

In [ ]:
# 평가지표 시각화 (yolo)
history = load_history("history_yolo.json")
plot_training_history(history, title_prefix="YOLO")

In [ ]:
# 평가지표 시각화 (faster rcnn)
history = load_history("history_frcnn.json")
plot_training_history(history, title_prefix="Faster_RCNN")

In [ ]:
# 평가지표 시각화 (retina)
history = load_history("history_retina.json")
plot_training_history(history, title_prefix="RetinaNet")

# loss 계산

In [ ]:
# YOLO loss 뽑기
train_loss = epoch_train_loss
val_loss = epoch_val_loss

In [ ]:
#Faster R-CNN loss 뽑기
loss_dict = model(images, targets)
losses = sum(loss for loss in loss_dict.values())
train_loss += losses.item()

In [ ]:
#RetinaNet loss 뽑기
loss_dict = model(images, targets)
losses = sum(loss for loss in loss_dict.values())
train_loss += losses.item()